In [ ]:
!pip install qiskit pennylane torch gymnasium numpy matplotlib

# Module 1: Custom Quantum Circuit RL Environment
This notebook constructs a Gymnasium-compliant Reinforcement Learning environment designed for Quantum Architecture Search (QAS) and circuit transpilation.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import torch
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, process_fidelity

class QuantumCircuitEnv(gym.Env):
    """
    Custom Gymnasium Environment for Quantum Circuit Depth Optimization via RL.
    """
    def __init__(self, num_qubits=3, max_depth=10):
        super(QuantumCircuitEnv, self).__init__()
        self.num_qubits = num_qubits
        self.max_depth = max_depth
        
        # Action space: 0=Keep, 1=Remove CNOT, 2=Merge Single-Qubit Gates
        self.action_space = spaces.Discrete(3)
        
        # Observation space: Flat array representing circuit gate indices and target qubits
        self.observation_space = spaces.Box(
            low=0, high=1, shape=(self.max_depth, self.num_qubits), dtype=np.float32
        )
        
        self.reset()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.target_circuit = self._generate_random_circuit()
        self.current_circuit = self.target_circuit.copy()
        self.target_unitary = Operator(self.target_circuit)
        self.step_count = 0
        
        obs = self._get_observation()
        return obs, {}

    def _generate_random_circuit(self):
        qc = QuantumCircuit(self.num_qubits)
        for _ in range(self.max_depth // 2):
            qc.h(np.random.randint(0, self.num_qubits))
            q1, q2 = np.random.choice(self.num_qubits, 2, replace=False)
            qc.cx(q1, q2)
            qc.rz(np.random.uniform(0, 2 * np.pi), np.random.randint(0, self.num_qubits))
        return qc

    def _get_observation(self):
        obs = np.zeros((self.max_depth, self.num_qubits), dtype=np.float32)
        for idx, instruction in enumerate(self.current_circuit.data[:self.max_depth]):
            for q in instruction.qubits:
                obs[idx, self.current_circuit.qubits.index(q)] = 1.0
        return obs

    def step(self, action):
        self.step_count += 1
        
        # Apply gate modification based on RL agent action
        if action == 1 and len(self.current_circuit.data) > 1:
            # Attempt CNOT gate removal/compression
            self.current_circuit.data.pop(np.random.randint(0, len(self.current_circuit.data)))
        elif action == 2 and len(self.current_circuit.data) > 1:
            # Attempt Single-qubit gate simplification
            self.current_circuit.data.pop(0)

        # Calculate unitary fidelity with the target circuit
        current_unitary = Operator(self.current_circuit)
        fidelity = process_fidelity(current_unitary, self.target_unitary)
        
        # Reward design: High fidelity reward + Penalty for circuit depth
        depth_penalty = 0.05 * self.current_circuit.depth()
        reward = (fidelity ** 2) - depth_penalty
        
        terminated = bool(fidelity > 0.99 or self.step_count >= self.max_depth)
        obs = self._get_observation()
        
        return obs, reward, terminated, False, {"fidelity": fidelity, "depth": self.current_circuit.depth()}

# Sanity Check
if __name__ == "__main__":
    env = QuantumCircuitEnv(num_qubits=3, max_depth=10)
    obs, _ = env.reset()
    print(f"[+] Environment initialized successfully. Observation shape: {obs.shape}")
    obs, reward, done, _, info = env.step(action=1)
    print(f"[+] Step Test -> Reward: {reward:.4f}, Fidelity: {info['fidelity']:.4f}, Depth: {info['depth']}")